# BioHub Fine-Tuning (RunPod Edition)
This notebook is optimized for RunPod. All data is saved to `/workspace` so it persists across restarts.

In [ ]:
import os
import json

print("Please paste your Kaggle credentials (make sure there are no spaces):")
username = input('KAGGLE_USERNAME: ').strip()
key = input('KAGGLE_KEY: ').strip()

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({"username": username, "key": key}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("Credentials saved successfully!")

In [ ]:
!apt-get update && apt-get install -y unzip
!pip install -q --upgrade kaggle zarr rustworkx networkx numcodecs

In [ ]:
import os

!mkdir -p /workspace/data /workspace/weights /workspace/repo
%cd /workspace

print("Downloading competition dataset...")
!kaggle competitions download -c biohub-cell-tracking-during-development -p data/
print("Extracting dataset (this will take a while)...")
!unzip -q -o data/biohub-cell-tracking-during-development.zip -d data/
print("Deleting large zip file to save disk space...")
!rm data/biohub-cell-tracking-during-development.zip

print("Downloading weights & repo...")
!kaggle datasets download pilkwang/biohub-tracking-support-pack-50ep-v1 -p weights/
!unzip -q -o weights/biohub-tracking-support-pack-50ep-v1.zip -d weights/
!rm weights/biohub-tracking-support-pack-50ep-v1.zip

if os.path.exists("weights/tracking_repo.zip"):
    !unzip -q -o weights/tracking_repo.zip -d repo/
elif os.path.exists("weights/tracking_repo"):
    !cp -r weights/tracking_repo/* repo/
print("Download and extraction complete!")

In [ ]:
import os
script_path = '/workspace/repo/scripts/train_unet_transformer.py'
if os.path.exists(script_path):
    with open(script_path, 'r') as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        if 'parser.add_argument("--unet-weights"' in line:
            lines.insert(i, '    parser.add_argument("--resume", type=str, default=None, help="Full checkpoint to resume.")\n')
            break
    for i, line in enumerate(lines):
        if 'model = UNetNodeTransformer(' in line:
            j = i
            while ').to(device)' not in lines[j]: j += 1
            lines.insert(j + 1, '''
    if getattr(args, "resume", None) is not None:
        import torch
        state = torch.load(args.resume, map_location="cpu", weights_only=True)
        state = {k.replace("unet.module.", "unet.", 1) if k.startswith("unet.") else k: v for k, v in state.items()}
        model.load_state_dict(state, strict=False)
        print(f"Resumed full model from {args.resume}", flush=True)
''')
            break
    with open(script_path, 'w') as f:
        f.writelines(lines)
    print("Training script patched successfully!")
else:
    print("Error: tracking_repo not found!")

In [ ]:
%cd /workspace
!pip install -e repo/

# Start Fine-Tuning!
!python repo/scripts/train_unet_transformer.py \
    --data-dir /workspace/data/train \
    --resume /workspace/weights/unet_transformer/split_0/checkpoint_last.pth \
    --epochs 10 \
    --lr 1e-5 \
    --batch-size 8 \
    --num-workers 8

print("\n✅ Training complete! Your fine-tuned weights are safely stored in /workspace/unet_transformer/")